In [1]:
import pandas as pd

# Load Dataset and Create Mapping

In [2]:
drug_mappings = pd.read_excel("/data/giobbi/CRESCENDDI/Data Record 4 - Drug mappings.xlsx")
drug_mappings = drug_mappings[drug_mappings["RXNORM_CODE/RXNORM_EXTENSION_CODE (OHDSI)"] != 0]
drug_mappings["DRUG_CONCEPT_NAME"] = drug_mappings["DRUG_CONCEPT_NAME"].str.lower()

emb_desc = pd.read_csv("/data/giobbi/embeddings/Dr_Desc_GPT.csv", sep="\t")
emb_desc = emb_desc[["Drug Name", "Drug ID"]]
emb_desc["Drug Name"] = emb_desc["Drug Name"].str.lower()

In [3]:
merged_mappings = drug_mappings.merge(
    emb_desc[["Drug Name", "Drug ID"]], left_on="DRUG_CONCEPT_NAME", right_on="Drug Name", how="left"
)

In [4]:
# 1. Get unmatched rows from the first merge
unmatched = merged_mappings[merged_mappings["Drug ID"].isna()].copy()

# 2. Prepare the column for matching
unmatched.drop(columns=["Drug Name", "Drug ID"], inplace=True)
unmatched["DRUG_INITIAL_NAME"] = unmatched["DRUG_INITIAL_NAME"].str.lower()

# 3. Merge again using DRUG_INITIAL_NAME
second_merge = unmatched.merge(
    emb_desc[["Drug Name", "Drug ID"]], left_on="DRUG_INITIAL_NAME", right_on="Drug Name", how="left"
)

# 4. Optionally, concatenate the results back to the original merged_mappings
# (excluding the previously unmatched rows)
final_merged = pd.concat([merged_mappings[~merged_mappings["Drug ID"].isna()], second_merge], ignore_index=True)

In [5]:
with pd.ExcelWriter("drug_mappings_and_emb_desc.xlsx") as writer:
    final_merged.to_excel(writer, sheet_name="MergedMappings", index=False)
    emb_desc.to_excel(writer, sheet_name="EmbDesc", index=False)

In [6]:
final_merged.to_csv("mapping_rxnorm_drugbank.csv", index=False)

In [7]:
final_merged[final_merged["Drug ID"].isna()]

,DRUG_INITIAL_NAME,DRUG_CONCEPT_NAME,RXNORM_CODE/RXNORM_EXTENSION_CODE (OHDSI),SOURCE,Drug Name,Drug ID
5158,epoetin alfa,epoetin alfa,1301125,BNF_DDI,NaN,NaN
5159,epoetin beta,epoetin beta,19001311,BNF_DDI,NaN,NaN
5160,epoetin zeta,epoetin zeta,36878697,BNF_DDI,NaN,NaN
5164,cannabis extract,cannabis sativa whole extract,1559806,BNF_DDI,NaN,NaN
5166,trientine,trientine,19004969,BNF_DDI,NaN,NaN
...,...,...,...,...,...,...
5906,anti-d (rh0) immunoglobulin,rho(d) immune globulin,535714,BNF_Single,NaN,NaN
5909,follitropin alfa,follitropin alfa,1542948,BNF_Single,NaN,NaN
5910,lithium citrate,lithium citrate,767410,BNF_Single,NaN,NaN
5911,follitropin delta,follicle stimulating hormone,1588712,BNF_Single,NaN,NaN


# Create Graph Data

In [8]:
pos_df = pd.read_excel("/data/giobbi/CRESCENDDI/Data Record 1 - Positive Controls.xlsx")
pos_df["DRUG_1_CONCEPT_NAME"] = pos_df["DRUG_1_CONCEPT_NAME"].str.lower()
pos_df["DRUG_2_CONCEPT_NAME"] = pos_df["DRUG_2_CONCEPT_NAME"].str.lower()
pos_df = pos_df[["DRUG_1_CONCEPT_NAME", "DRUG_2_CONCEPT_NAME"]]
pos_df.loc[:, "label"] = 1


neg_df = pd.read_excel("/data/giobbi/CRESCENDDI/Data Record 2 - Negative Controls.xlsx")
neg_df["DRUG_1_CONCEPT_NAME"] = neg_df["DRUG_1_CONCEPT_NAME"].str.lower()
neg_df["DRUG_2_CONCEPT_NAME"] = neg_df["DRUG_2_CONCEPT_NAME"].str.lower()
neg_df = neg_df[["DRUG_1_CONCEPT_NAME", "DRUG_2_CONCEPT_NAME"]]
neg_df.loc[:, "label"] = 0

edges = pd.concat([pos_df, neg_df], ignore_index=True)

In [9]:
name_to_id = final_merged.set_index("DRUG_CONCEPT_NAME")["Drug ID"].to_dict()  # ['Drug ID'].to_dict()

edges["Drug1"] = edges["DRUG_1_CONCEPT_NAME"].map(name_to_id)
edges["Drug2"] = edges["DRUG_2_CONCEPT_NAME"].map(name_to_id)

In [19]:
pos_edges = edges[edges["label"] == 1]
pos_edges = pos_edges[~(pos_edges["Drug1"].isna() | pos_edges["Drug2"].isna())]
GRAPH_URL = "/data/giobbi/CRESCENDDI/positive_edges_CRESCENDDI.csv"
pos_edges[["Drug1", "Drug2"]].to_csv(GRAPH_URL, index=False, sep="\t")

## Anaysis of missing drug ids

In [ ]:
edges[edges["DRUG_1_ID"].isna() | edges["DRUG_2_ID"].isna()]

,DRUG_1_CONCEPT_NAME,DRUG_2_CONCEPT_NAME,label,DRUG_1_ID,DRUG_2_ID
412,aspirin,nabumetone,1,NaN,DB00461
413,aspirin,naproxen,1,NaN,DB00788
414,aspirin,celecoxib,1,NaN,DB00482
415,aspirin,diclofenac,1,NaN,DB00586
416,aspirin,piroxicam,1,NaN,DB00554
...,...,...,...,...,...
14666,losartan,ciprofibrate,0,DB00678,NaN
14695,valproate,tolterodine,0,NaN,DB01036
14791,amantadine,lithium,0,DB00915,NaN
14793,valproate,glyburide,0,NaN,DB01016


In [ ]:
# get missing drug ids from edges with drug names and occurence count
missing_drug_1 = edges[edges["DRUG_1_ID"].isna()]["DRUG_1_CONCEPT_NAME"].value_counts()
missing_drug_2 = edges[edges["DRUG_2_ID"].isna()]["DRUG_2_CONCEPT_NAME"].value_counts()

# stack df and group by index to sum counts as we don't care if missing in drug 1 or drug 2, sort descending
pd.concat([missing_drug_1, missing_drug_2], axis=1, keys=["Missing_DRUG_1_Count", "Missing_DRUG_2_Count"]).fillna(
    0
).astype(int).sum(axis=1).sort_values(ascending=False)

aspirin                  123
lithium                  115
valproate                 46
trimeprazine              33
atracurium                31
dothiepin                 19
canrenoate                18
ciprofibrate              14
polystyrene sulfonate     13
flupenthixol               2
dtype: int64